# CONVERT MiniSEED/SAC to SDS

SmartSolo data are recorded in a proprietary format and harvested with the SoloLite.exe software.
Next then are converted to MiniSEED (though I mistakenly converted one directory to SAC), also using SoloLite.
To run MSNoise, we want to organize them in an SDS archive. We change the network and location codes to make identification easier.

In [ ]:
from pathlib import Path
import re
from obspy import read

base = Path("/Volumes/tachyon/LBSSP_DATA/SOLODATA/prospect_geokarst/Karst_Geophysics")
dst = Path("/Volumes/tachyon/LBSSP_DATA/nodal_sds")

# These are the different deployments converted with SoloLite software. The keys are the folder names, and the values are tuples of (network, location) codes to assign to the traces.
deployments = {
    "Transect1_Nodal1_500Hz":  ("T1", "N1"),
    "Transect1_Nodal2_500Hz":  ("T1", "N2"),
    "Transect1_Nodal3_500Hz":  ("T1", "N3"),
    "TransectE_Nodal4_500Hz":  ("T3", "N4"),
    "Transect1_Nodal1_1000Hz": ("T1", "N1"),
    "Transect1_Nodal2_1000Hz": ("T1", "N2"),
    "Transect1_Nodal3_1000Hz": ("T1", "N3"),
    "TransectE_Nodal4_1000Hz": ("T3", "N4"),
}

pattern = re.compile(
    r"(?P<serial>\d+)\.(?P<fileno>\d+)\."
    r"(?P<year>\d{4})\.(?P<month>\d{2})\.(?P<day>\d{2})\."
    r"(?P<hour>\d{2})\.(?P<minute>\d{2})\.(?P<second>\d{2})\."
    r"(?P<msec>\d{3})\.(?P<comp>[ENZ])\.miniseed$"
)

for folder, (net, loc) in deployments.items():
    src = base / folder

    if not src.exists():
        print(f"Missing folder, skipping: {src}")
        continue

    print(f"\nProcessing {src}")
    print(f"  Network={net}, Location={loc}")

    for f in sorted(src.glob("*.miniseed")):
        m = pattern.match(f.name)
        if not m:
            print("  Skipping:", f.name)
            continue

        try:
            st = read(str(f))
        except Exception as e:
            print(f"  Could not read {f.name}: {e}")
            continue

        for tr in st:
            # Preserve SmartSolo station and channel codes.
            # Only replace network and location.
            tr.stats.network = net
            tr.stats.location = loc

            year = tr.stats.starttime.year
            doy = tr.stats.starttime.julday
            sta = tr.stats.station
            cha = tr.stats.channel

            outdir = dst / f"{year}" / net / sta / f"{cha}.D"
            outdir.mkdir(parents=True, exist_ok=True)

            outfile = outdir / f"{net}.{sta}.{loc}.{cha}.D.{year}.{doy:03d}"

            if outfile.exists():
                print(f"  Warning: overwriting {outfile}")

            tr.write(str(outfile), format="MSEED")

print(f"\nDone: {dst}")